# NegotiateAI — GRPO Training Notebook
### Meta PyTorch OpenEnv Hackathon | April 2026

This notebook trains a Llama 3.2 3B model to negotiate enterprise procurement contracts
using Group Relative Policy Optimisation (GRPO). The model learns entirely from reward
signals returned by the live NegotiateAI environment — no human labels required.

**Pipeline overview:**
1. Connect to the live NegotiateAI environment on HuggingFace Spaces
2. Generate training episodes via random exploration (collect prompt → reward pairs)
3. Fine-tune Llama 3.2 3B using GRPO — the model learns to prefer actions that score higher
4. Plot reward curves showing improvement over training
5. Run a before/after comparison to demonstrate what the agent learned

**Runtime required:** GPU (T4 or better) — set via Runtime → Change runtime type

**Estimated total time on T4:**
- Episode generation: ~34 minutes (200 episodes)
- GRPO training: ~6 hours (2116 samples, 3 epochs)
- Total: ~6.5 hours

**Environment:** [HuggingFace Space](https://huggingface.co/spaces/prasanthdj8/negotiateai-openenv) | [GitHub](https://github.com/Prasanthdj8/negotiateai-openenv)

---

## Cell 1 — Install Dependencies

Installs the exact versions validated during dry run. The dependency resolver will show
some warnings about unsloth-zoo — these are safe to ignore; they relate to packages
already present in the Colab environment that we don't use.

**After this cell completes:** Runtime → Restart session, then continue from Cell 2.

In [ ]:
# Pinned versions — validated on T4, April 2026
!pip install trl==0.22.0 -q
!pip install peft==0.14.0 -q
!pip install transformers==4.55.0 datasets==3.4.1 accelerate==1.6.0 -q
!pip install pydantic==2.10.6 bitsandbytes -q
!pip install requests numpy matplotlib -q
print("✅ Installation complete — restart session before continuing")

### Patch TRL GRPOConfig (run after restart)

Colab ships a pre-installed version of TRL that has a bug in `GRPOConfig` — it raises
a `ValueError` when `generation_batch_size` and `steps_per_generation` are both set,
even when only one is explicitly configured. This cell patches the installed file directly
to resolve the conflict gracefully.

In [ ]:
# Patch the Colab-preinstalled TRL to fix GRPOConfig conflict
grpo_config_path = "/usr/local/lib/python3.12/dist-packages/trl/trainer/grpo_config.py"

with open(grpo_config_path, "r") as f:
    content = f.read()

old = """        else:
            raise ValueError(
                "'generation_batch_size' and 'steps_per_generation' can not be both configured at the same time"
            )"""

new = """        else:
            # Resolve conflict by deriving steps_per_generation from generation_batch_size
            self.steps_per_generation = self.generation_batch_size // (self.per_device_train_batch_size * num_processes)"""

content = content.replace(old, new)

with open(grpo_config_path, "w") as f:
    f.write(content)

print("✅ TRL GRPOConfig patched" if new.strip() in content else "❌ Patch failed — check file manually")

## Cell 2 — Configuration

Set your credentials and training hyperparameters here.

**Before running:** Replace `OPENAI_KEY` and `HF_TOKEN` with your own keys.
- OpenAI key: used by the NegotiateAI environment to power supplier LLMs
- HF token: needed for Cell 12 (push model to Hub) — optional

**Training config notes:**
- `NUM_EPOCHS=3` and `EASY_EPISODES=200` give the full training run (~6 hrs on T4)
- `BATCH_SIZE=4` is set to match `NUM_GENERATIONS=4` — required by GRPOTrainer
- Lower `NUM_EPOCHS=1` and `EASY_EPISODES=50` for a quick smoke test (~30 mins)

In [ ]:
import os

# ── CREDENTIALS — replace with your own ───────────────────────────────────────
ENV_URL    = "https://prasanthdj8-negotiateai-openenv.hf.space"
OPENAI_KEY = "your-openai-key-here"
HF_TOKEN   = "your-hf-token-here"   # only needed for Cell 12

# ── MODEL ─────────────────────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True    # 4-bit quantisation fits on a free T4 (15GB VRAM)

# ── TRAINING HYPERPARAMETERS ───────────────────────────────────────────────────
LEARNING_RATE    = 5e-6
NUM_EPOCHS       = 3     # full run; set to 1 for a quick smoke test
BATCH_SIZE       = 4     # must equal NUM_GENERATIONS for GRPOTrainer
NUM_GENERATIONS  = 4     # candidate actions generated per GRPO step
MAX_COMPLETION_LEN = 300
LOGGING_STEPS    = 10

# ── EPISODE GENERATION ────────────────────────────────────────────────────────
EASY_EPISODES    = 200   # ~34 min on T4; produces ~2116 training samples
MEDIUM_EPISODES  = 100   # optional — only used in Cell 11
MAX_STEPS_PER_EP = 30

SAVE_DIR = "/content/negotiateai-grpo"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ Config ready")
print(f"   ENV_URL:     {ENV_URL}")
print(f"   Model:       {MODEL_NAME}")
print(f"   Batch size:  {BATCH_SIZE}")
print(f"   Generations: {NUM_GENERATIONS}")
print(f"   Epochs:      {NUM_EPOCHS}")

## Cell 3 — Verify Environment

Confirms the NegotiateAI HuggingFace Space is live and responding before training begins.
Also defines the environment helper functions used throughout the notebook.

If the health check fails, visit the Space URL in a browser first to wake it from sleep,
then re-run this cell.

In [ ]:
import requests, json

# ── Environment API wrappers ───────────────────────────────────────────────────

def env_health():
    """Returns True if the HF Space is live and healthy."""
    try:
        r = requests.get(f"{ENV_URL}/health", timeout=15)
        return r.status_code == 200 and r.json().get("status") == "healthy"
    except Exception as e:
        print(f"  Health check error: {e}")
        return False

def env_reset(task_id="easy_negotiation", seed=None):
    """Start a new episode. Returns observation dict."""
    r = requests.post(f"{ENV_URL}/reset",
        json={"task_id": task_id, "seed": seed}, timeout=30)
    r.raise_for_status()
    return r.json()

def env_step(action):
    """
    Submit an action and receive the next observation and reward.
    A 400 response means the episode has reached a terminal state —
    we treat this as a zero-reward episode end rather than a crash.
    """
    r = requests.post(f"{ENV_URL}/step",
        json={"action": action}, timeout=30)
    if r.status_code == 400:
        return {"reward": {"total": 0.0}, "observation": {"done": True}, "done": True}
    r.raise_for_status()
    return r.json()

def env_tasks():
    """List all available tasks with baseline and target scores."""
    r = requests.get(f"{ENV_URL}/tasks", timeout=15)
    return r.json()

# ── Health check ───────────────────────────────────────────────────────────────
print("Checking environment health...")
if env_health():
    print("✅ Environment is healthy!")
else:
    print("❌ Environment not reachable.")
    print("   Visit the HF Space URL in a browser first to wake it, then re-run.")

tasks_data = env_tasks()
print(f"\n📋 Available tasks ({tasks_data['count']}):")
for t in tasks_data["tasks"]:
    print(f"   {t['task_id']:30s} baseline={t['baseline_score']} → target={t['target_score']}")

print("\nRunning quick reset test...")
reset_data = env_reset("easy_negotiation", seed=42)
obs = reset_data["observation"]
print(f"✅ Reset OK: week={obs['week']}/{obs['total_weeks']} "
      f"suppliers={len(obs['suppliers'])} "
      f"budget=${obs['budget_remaining']:,.0f}")

## Cell 4 — Prompt Templates

Defines the two prompts used throughout training and inference:

- **System prompt:** Instructs the model on its role, strategy rules, and the exact JSON
  output format it must produce. The strategy guide is intentionally opinionated — it
  encodes domain knowledge about procurement (e.g. negotiate below list price, escalate
  near deadlines, hedge when rival pressure is high) so the model starts from a reasonable
  prior rather than learning from scratch.

- **User prompt (`obs_to_prompt`):** Converts the raw observation dict from the environment
  into a compact, human-readable situation summary. This is what the model sees each step.

In [ ]:
SYSTEM_PROMPT = """You are an expert AI procurement agent for a tech company.
Your goal: source software, hardware, and services at the best cost,
within budget, before deadlines — while outperforming a rival buyer
and detecting deceptive suppliers.

STRATEGY GUIDE:
1. Always negotiate price DOWN from list — open below list price
2. Raise a PR before awarding contracts (compliance requirement)
3. Watch rival_pressure > 0.6 — hedge orders if this happens
4. Reject suppliers with community_rating < 0.65 or known issues
5. Escalate PRs when deadline ≤ 2 weeks away
6. Hedge critical items across 2 suppliers when rival pressure is high
7. Read supplier messages carefully — deceptive suppliers deflect on reliability

Respond with ONLY valid JSON. No markdown. No preamble:
{
  "action_type": "negotiate|award_contract|reject|raise_pr|escalate|hedge|defer|cancel_contract",
  "supplier_id": "<from observation>",
  "item_id": "<from requirements>",
  "message": "<natural language to supplier>",
  "proposed_price": <float or null>,
  "proposed_quantity": <int or null>,
  "proposed_lead_time": <int or null>,
  "hedge_supplier_id": "<second supplier for hedge or null>",
  "hedge_quantity": <int or null>,
  "notes": "<your reasoning>"
}""".strip()


def obs_to_prompt(obs):
    """Convert an environment observation dict into a structured LLM prompt."""
    suppliers    = obs.get("suppliers", [])
    requirements = obs.get("requirements", [])
    contracts    = obs.get("contracts", [])
    signals      = obs.get("market_signals", [])[-3:]
    disruptions  = obs.get("disruptions", [])
    rival_activity = obs.get("rival_activity", {})

    sup_lines = []
    for s in suppliers:
        status = "ACTIVE" if s["is_active"] else "OFFLINE"
        flag = " ⚠️RIVAL" if s.get("rival_pressure", 0) > 0.6 else ""
        sup_lines.append(
            f"  {s['supplier_id']} | {s['name']} ({s['category']}) | "
            f"${s['base_price']:.0f}/unit | lead={s['lead_time_days']}d | "
            f"rel={s['reliability_score']:.2f} | cap={s['capacity_available']} | "
            f"{status}{flag}"
        )

    req_lines = []
    for r in requirements:
        done = "DONE" if r["fulfilled"] else "PENDING"
        crit = "[CRITICAL]" if r["is_critical"] else ""
        wleft = r["deadline_week"] - obs.get("week", 1)
        req_lines.append(
            f"  {r['item_id']} | {r['name']} | qty={r['quantity']} | "
            f"deadline=wk{r['deadline_week']}({wleft}w left) | "
            f"budget=${r['budget_ceiling']:,.0f} | {done} {crit}"
        )

    sig_lines = [f"  [{s['signal_type']}] {s['description'][:70]}" for s in signals]
    dis_lines = [f"  ⚡ {d['description']}" for d in disruptions]
    rival_high = [
        f"{sid}(p={p:.2f})"
        for sid, p in rival_activity.items() if p > 0.4
    ]

    return f"""WEEK {obs.get('week','?')}/{obs.get('total_weeks','?')} | BUDGET ${obs.get('budget_remaining',0):,.0f} / ${obs.get('budget_total',0):,.0f}

REQUIREMENTS:
{chr(10).join(req_lines) or '  None'}

SUPPLIERS:
{chr(10).join(sup_lines) or '  None'}

CONTRACTS ACTIVE: {len([c for c in contracts if c.get('status') == 'active'])}

SIGNALS:
{chr(10).join(sig_lines) or '  None'}

DISRUPTIONS:
{chr(10).join(dis_lines) or '  None'}

RIVAL HIGH PRESSURE: {', '.join(rival_high) or 'Low'}
RIVAL CONTRACTS WON: {obs.get('rival_contracts_won', 0)}

Respond with JSON action only.""".strip()


print("✅ Prompts ready")
print(f"   System prompt: {len(SYSTEM_PROMPT)} chars")
reset_data = env_reset("easy_negotiation", seed=1)
test_prompt = obs_to_prompt(reset_data["observation"])
print(f"   Sample prompt: {len(test_prompt)} chars")
print("   Preview:")
print("   " + "\n   ".join(test_prompt.split("\n")[:5]))

## Cell 5 — Generate Training Episodes

Runs the environment with random actions to collect (prompt, reward) pairs for GRPO training.
This is the exploration phase — GRPO requires diverse trajectories to learn from, so random
action selection is intentional here.

**What this produces:** Each episode generates ~10 training samples. 200 episodes → ~2116 samples.

**Time:** ~34 minutes on T4 for 200 episodes.

The dataset is automatically saved to Google Drive so it survives session restarts —
you never need to regenerate it. On subsequent runs, skip to Cell 5b to load from Drive.

In [ ]:
import random
import numpy as np

ACTION_TYPES = [
    "negotiate", "raise_pr", "award_contract",
    "defer", "reject", "hedge", "escalate"
]

def random_action(obs):
    """Sample a random valid action from the current observation."""
    suppliers    = [s for s in obs.get("suppliers", []) if s["is_active"]]
    requirements = [r for r in obs.get("requirements", []) if not r["fulfilled"]]

    if not suppliers:    suppliers    = obs.get("suppliers", [])
    if not requirements: requirements = obs.get("requirements", [])
    if not suppliers or not requirements:
        return None

    sup   = random.choice(suppliers)
    req   = random.choice(requirements)
    atype = random.choice(ACTION_TYPES)

    action = {
        "action_type":       atype,
        "supplier_id":       sup["supplier_id"],
        "item_id":           req["item_id"],
        "message":           f"Exploring {atype} with {sup['name']} for {req['name']}",
        "proposed_price":    round(sup["base_price"] * random.uniform(0.80, 1.05), 2),
        "proposed_quantity": req["quantity"],
        "notes":             "Random exploration episode",
    }

    # Hedge actions require a second supplier
    if atype == "hedge" and len(suppliers) >= 2:
        other = random.choice([s for s in suppliers if s["supplier_id"] != sup["supplier_id"]])
        action["hedge_supplier_id"] = other["supplier_id"]
        action["hedge_quantity"]    = max(1, req["quantity"] // 2)

    return action


def generate_episodes(task_id="easy_negotiation", n_episodes=200, max_steps=30, seed=42):
    """
    Generate training episodes via random exploration.

    For each episode:
    - Reset the environment to a fresh state
    - Take up to max_steps random actions
    - Record (system_prompt, user_prompt) pairs with their corresponding rewards

    Returns a list of dicts suitable for conversion to a HuggingFace Dataset.
    """
    random.seed(seed)
    dataset = []
    episode_scores = []

    print(f"Generating {n_episodes} episodes for {task_id}...")
    print(f"  Max steps per episode: {max_steps}")

    for ep in range(n_episodes):
        try:
            reset_data = env_reset(task_id, seed=seed + ep)
            obs = reset_data["observation"]
            episode_rewards = []

            for step_i in range(max_steps):
                if obs.get("done", False):
                    break

                action = random_action(obs)
                if action is None:
                    break

                # Capture the prompt BEFORE stepping — this is what the model will learn from
                prompt = obs_to_prompt(obs)

                step_data = env_step(action)
                reward    = step_data["reward"]["total"]
                obs       = step_data["observation"]
                episode_rewards.append(reward)

                dataset.append({
                    "prompt": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    "reward":  reward,
                    "task_id": task_id,
                    "episode": ep,
                    "step":    step_i,
                })

            if episode_rewards:
                episode_scores.append(np.mean(episode_rewards))

            if (ep + 1) % 20 == 0:
                avg = np.mean(episode_scores[-20:]) if episode_scores else 0
                print(f"  Episode {ep+1:4d}/{n_episodes} | avg_reward={avg:.4f} | dataset_size={len(dataset)}")

        except Exception as e:
            print(f"  Episode {ep}: error — {e}")
            continue

    print(f"\n✅ Generated {len(dataset)} training samples from {n_episodes} episodes")
    print(f"   Reward stats: mean={np.mean(episode_scores):.4f} "
          f"min={np.min(episode_scores):.4f} "
          f"max={np.max(episode_scores):.4f}")
    return dataset, episode_scores


easy_dataset, easy_scores = generate_episodes(
    task_id="easy_negotiation",
    n_episodes=EASY_EPISODES,
    max_steps=MAX_STEPS_PER_EP,
    seed=42,
)
print(f"\n📊 Dataset ready: {len(easy_dataset)} samples")

### Cell 5a — Save Dataset to Google Drive

Run this immediately after episode generation to persist the dataset across sessions.
Colab runtimes reset after ~12 hours — saving to Drive means you never regenerate 2116 episodes.

In [ ]:
import json, shutil
from google.colab import drive

# Save locally first
with open("/content/easy_dataset.json", "w") as f:
    json.dump(easy_dataset, f)
print(f"✅ Saved {len(easy_dataset)} samples to /content/easy_dataset.json")

# Back up to Google Drive
drive.mount("/content/drive")
shutil.copy("/content/easy_dataset.json", "/content/drive/MyDrive/easy_dataset.json")
print("✅ Backed up to Google Drive — safe across session restarts")

### Cell 5b — Load Dataset from Google Drive (skip Cell 5 and 5a on subsequent runs)

On any run after the first, skip straight here to load the pre-generated dataset.
This saves 34 minutes and avoids unnecessary API calls to the environment.

In [ ]:
import json
from google.colab import drive

drive.mount("/content/drive")

with open("/content/drive/MyDrive/easy_dataset.json", "r") as f:
    easy_dataset = json.load(f)

print(f"✅ Loaded {len(easy_dataset)} samples from Google Drive")

## Cell 6 — Load Model

Loads Llama 3.2 3B Instruct in 4-bit quantisation using HuggingFace Transformers + PEFT.
4-bit quantisation reduces VRAM from ~12GB to ~4GB, leaving headroom for activations
and GRPO's candidate generation.

LoRA (Low-Rank Adaptation) is applied to the attention layers — only 9.2M of 1.8B
parameters are trained, keeping GPU memory usage manageable on a free T4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

print(f"GPU:  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — check runtime type'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"\nLoading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True,
)

# Apply LoRA — train only attention projection layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

print("\n✅ Model loaded")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   Total params:     {sum(p.numel() for p in model.parameters()):,}")

test_tokens = tokenizer("Test procurement action", return_tensors="pt")
print(f"   Tokenizer OK:     {test_tokens['input_ids'].shape}")

## Cell 7 — GRPO Reward Function

This is the core of training. At each GRPO step, the trainer generates `NUM_GENERATIONS`
candidate actions for each prompt, then calls this function to score them.
The model is updated to increase the probability of higher-scoring actions.

**Reward signal flow:**
1. Parse the LLM's JSON output
2. Reset the environment to a fresh episode
3. Submit the action and receive a reward from the environment
4. Return the reward — GRPO handles the policy gradient update

**Penalty:** Invalid JSON or missing required fields → floor reward of 1e-4.
This teaches the model to always output valid, structured actions.

In [ ]:
import math
import numpy as np
import json
import random
import requests

# Redefine env wrappers here so this cell is self-contained after a session restart
def env_reset(task_id="easy_negotiation", seed=None):
    r = requests.post(f"{ENV_URL}/reset",
        json={"task_id": task_id, "seed": seed}, timeout=30)
    r.raise_for_status()
    return r.json()

def env_step(action):
    r = requests.post(f"{ENV_URL}/step",
        json={"action": action}, timeout=10)   # 10s hard timeout per step
    if r.status_code == 400:
        return {"reward": {"total": 0.0}, "observation": {"done": True}, "done": True}
    r.raise_for_status()
    return r.json()

# Reward tracking — used for plotting in Cell 9
grpo_step_rewards = []
grpo_rolling_avgs = []


def procurement_reward_fn(completions, prompts=None, **kwargs):
    """
    Reward function called by GRPOTrainer at each training step.

    For each completion (candidate action from the model):
    - Parse the JSON action
    - Validate required fields; fall back to observation defaults if needed
    - Reset the environment and step with the action
    - Return the reward, clamped to (1e-4, 1-1e-4)

    Invalid JSON receives the minimum reward of 1e-4, penalising malformed outputs.
    """
    rewards = []

    for completion in completions:
        # Extract text from the completion object
        if isinstance(completion, list):
            text = completion[0].get("content", "") if completion else ""
        else:
            text = str(completion)

        # Strip markdown fences if present
        text = text.strip()
        if text.startswith("```"):
            parts = text.split("```")
            text = parts[1] if len(parts) > 1 else text
            if text.startswith("json"):
                text = text[4:].strip()

        try:
            action = json.loads(text)

            # Required fields check — penalise if missing
            if not all(k in action for k in ["action_type", "supplier_id", "item_id", "message"]):
                rewards.append(1e-4)
                continue

            # Reset environment and validate IDs against current observation
            reset_data  = env_reset("easy_negotiation", seed=random.randint(0, 999))
            obs         = reset_data["observation"]
            valid_sups  = {s["supplier_id"] for s in obs["suppliers"]}
            valid_items = {r["item_id"]      for r in obs["requirements"]}

            # Correct invalid IDs rather than penalising — the action type still matters
            if action["supplier_id"] not in valid_sups:
                action["supplier_id"] = obs["suppliers"][0]["supplier_id"]
            if action["item_id"] not in valid_items:
                action["item_id"] = obs["requirements"][0]["item_id"]

            step_data = env_step(action)
            reward    = float(np.clip(step_data["reward"]["total"], 1e-4, 1 - 1e-4))
            rewards.append(reward)

        except json.JSONDecodeError:
            rewards.append(1e-4)
        except Exception:
            rewards.append(1e-4)

    if rewards:
        grpo_step_rewards.extend(rewards)
        grpo_rolling_avgs.append(np.mean(rewards))

    return rewards


# ── Pre-training JSON sanity check ────────────────────────────────────────────
# Verify the base model can generate valid JSON before wasting compute on GRPO.
# If valid_count < 2, add a few hand-written SFT examples before running Cell 8.
print("Running pre-training sanity check...")
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                max_new_tokens=200, temperature=0.3, do_sample=True,
                pad_token_id=tokenizer.eos_token_id)

reset_data = env_reset("easy_negotiation", seed=42)
obs        = reset_data["observation"]
test_prompt = obs_to_prompt(obs)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_prompt},
]
text_input = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

valid_count = 0
for attempt in range(5):
    try:
        out     = pipe(text_input)[0]["generated_text"]
        # Extract only the generated part
        gen     = out[len(text_input):].strip()
        if gen.startswith("```"):
            gen = gen.split("```")[1].lstrip("json").strip()
        action  = json.loads(gen)
        if all(k in action for k in ["action_type", "supplier_id", "item_id", "message"]):
            valid_count += 1
    except Exception:
        pass

print(f"   Valid JSON generations: {valid_count}/5")
if valid_count >= 2:
    print("   ✅ Base model can generate valid actions — safe to run GRPO")
elif valid_count == 1:
    print("   ⚠️  Only 1/5 valid — GRPO may be slow to start. Consider running 20 SFT steps first.")
else:
    print("   ❌ 0/5 valid JSON — DO NOT run GRPO yet.")
    print("      The model needs JSON format priming before RL will work.")
    print("      Add 10-20 hand-written (prompt, action) SFT examples and run SFT for 1 epoch first.")

del pipe  # free VRAM before training
import gc; gc.collect()
if hasattr(torch.cuda, "empty_cache"):
    torch.cuda.empty_cache()

# ── Reward function test ───────────────────────────────────────────────────────
print("\nTesting reward function...")
test_completions = [
    [{"content": json.dumps({
        "action_type": "negotiate",
        "supplier_id": "sup_001",
        "item_id": "item_001",
        "message": "We need 50 licenses at $950 each",
        "proposed_price": 950.0,
        "proposed_quantity": 50,
        "notes": "Opening below list price"
    })}],
    [{"content": "invalid json here"}],
]
test_rewards = procurement_reward_fn(test_completions)
print(f"✅ Reward function working")
print(f"   Valid action reward:  {test_rewards[0]:.4f}  (expected > 0.01)")
print(f"   Invalid JSON reward:  {test_rewards[1]:.4f}  (expected 0.0001)")

## Cell 8 — GRPO Training

Fine-tunes the model using Group Relative Policy Optimisation (GRPO).

At each step, the trainer:
1. Samples a batch of prompts from the dataset
2. Generates `NUM_GENERATIONS=4` candidate action completions per prompt
3. Scores each completion using `procurement_reward_fn` (live environment calls)
4. Computes relative advantages within the group and updates the model

The key insight of GRPO over PPO: no separate value network is needed. The baseline
is computed from the group mean, making it more memory-efficient for LLM fine-tuning.

**Expected training time:** ~6 hours on T4 for the full dataset (2116 samples, 3 epochs)

Watch the loss in the training table — GRPO loss is expected to be negative and noisy.
The meaningful signal is the reward curve in Cell 9, not the loss value.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

# Build HuggingFace Dataset from the generated episodes
# Each sample contains the (system, user) prompt pair — the model generates the action
hf_dataset = Dataset.from_list([
    {"prompt": ep["prompt"]}
    for ep in easy_dataset
])

print(f"Training dataset: {len(hf_dataset)} samples")
print(f"First sample prompt preview:")
print(hf_dataset[0]["prompt"][1]["content"][:200] + "...")

grpo_config = GRPOConfig(
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,   # must equal num_generations
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    generation_batch_size=NUM_GENERATIONS,    # explicit to avoid TRL version conflicts
    max_completion_length=MAX_COMPLETION_LEN,
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION_LEN,
    output_dir=SAVE_DIR,
    logging_steps=LOGGING_STEPS,
    save_steps=50,
    warmup_ratio=0.05,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    fp16=True,    # T4 uses fp16 (not bf16)
    bf16=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=procurement_reward_fn,
    args=grpo_config,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
)

print(f"\n🚀 Starting GRPO training...")
print(f"   Model:       {MODEL_NAME}")
print(f"   Dataset:     {len(hf_dataset)} samples")
print(f"   Epochs:      {NUM_EPOCHS}")
print(f"   Batch size:  {BATCH_SIZE}")
print(f"   Generations: {NUM_GENERATIONS}")
print(f"   LR:          {LEARNING_RATE}")
print()

trainer.train()

print("\n✅ Training complete!")
print(f"   Steps completed: {trainer.state.global_step}")

model.save_pretrained(f"{SAVE_DIR}/final")
tokenizer.save_pretrained(f"{SAVE_DIR}/final")
print(f"   Model saved to:  {SAVE_DIR}/final")

## Cell 9 — Training Results

Plots the reward progression and training loss from the completed run.

- **Left chart:** Step-level rewards (scatter) and rolling average (line), with reference
  lines for the random baseline (0.15) and training target (0.60). An upward trend in the
  rolling average confirms the model is learning to take better actions.

- **Right chart:** GRPO training loss per logging step. GRPO loss is typically negative
  and noisy — this is expected behaviour, not a sign of divergence.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

log_history = trainer.state.log_history
steps  = [l["step"] for l in log_history if "loss" in l]
losses = [l["loss"] for l in log_history if "loss" in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("NegotiateAI — GRPO Training Results", fontsize=14, fontweight="bold")

# ── Reward progression ─────────────────────────────────────────────────────────
ax1 = axes[0]
if grpo_step_rewards:
    window  = 20
    rolling = [np.mean(grpo_step_rewards[max(0,i-window):i+1]) for i in range(len(grpo_step_rewards))]

    ax1.scatter(range(len(grpo_step_rewards)), grpo_step_rewards,
                alpha=0.2, s=8, color="#94a3b8", label="Step reward")
    ax1.plot(rolling, color="#3b82f6", linewidth=2, label=f"Rolling avg (n={window})")
    ax1.axhline(y=0.15, color="#ef4444", linestyle="--", linewidth=1.5,
                label="Random baseline (0.15)", alpha=0.7)
    ax1.axhline(y=0.60, color="#22c55e", linestyle="--", linewidth=1.5,
                label="Target (0.60)", alpha=0.7)
    ax1.set_xlabel("Training Step")
    ax1.set_ylabel("Reward")
    ax1.set_title("Step-Level Reward Progression")
    ax1.legend(fontsize=9)
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)

# ── Training loss ──────────────────────────────────────────────────────────────
ax2 = axes[1]
if steps and losses:
    ax2.plot(steps, losses, color="#8b5cf6", linewidth=2)
    ax2.set_xlabel("Training Step")
    ax2.set_ylabel("Loss")
    ax2.set_title("Training Loss (GRPO)")
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "Loss data not available",
             ha="center", va="center", transform=ax2.transAxes, fontsize=12)
    ax2.set_title("Training Loss")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

if grpo_step_rewards and len(grpo_step_rewards) >= 40:
    first_20    = np.mean(grpo_step_rewards[:20])
    last_20     = np.mean(grpo_step_rewards[-20:])
    improvement = ((last_20 - first_20) / max(first_20, 1e-6)) * 100
    print(f"\n📊 Training Summary:")
    print(f"   First 20 steps avg reward:  {first_20:.4f}")
    print(f"   Last  20 steps avg reward:  {last_20:.4f}")
    print(f"   Improvement:                +{improvement:.1f}%")
    print(f"   Chart saved: {SAVE_DIR}/reward_curve.png")

# ── Component reward breakdown ─────────────────────────────────────────────────
# Fetch per-component trends from the curriculum endpoint — shows judges
# that individual reward signals (not just total) are all improving.
print("\n📊 Fetching component reward breakdown...")
try:
    curve_data = requests.get(
        f"{ENV_URL}/curriculum/easy_negotiation/curve", timeout=10
    ).json()

    curve        = curve_data["curve"]
    episodes     = [c["episode"]     for c in curve]
    scores       = [c["score"]       for c in curve]
    rolling_avgs = [c["rolling_avg"] for c in curve]
    tiers        = [c["tier"]        for c in curve]

    # Tier advancement markers
    advancements = [(c["episode"], c["tier"]) for c in curve if c.get("advanced")]

    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle("NegotiateAI — Curriculum Progression", fontsize=14, fontweight="bold")

    # Left: curriculum reward curve with tier markers
    ax3 = axes2[0]
    ax3.plot(episodes, scores,       alpha=0.3, color="#94a3b8", linewidth=1, label="Episode score")
    ax3.plot(episodes, rolling_avgs, color="#3b82f6", linewidth=2, label="Rolling avg")

    tier_colors = {"novice": "#94a3b8", "apprentice": "#60a5fa",
                   "practitioner": "#a78bfa", "expert": "#f59e0b", "master": "#ef4444"}
    for ep, tier in advancements:
        color = tier_colors.get(tier, "#000")
        ax3.axvline(x=ep, color=color, linestyle="--", alpha=0.7, linewidth=1.5)
        ax3.text(ep+1, 0.85, f"→{tier[:3]}", fontsize=7, color=color, rotation=90)

    ax3.set_xlabel("Episode")
    ax3.set_ylabel("Score")
    ax3.set_title("Curriculum Reward Curve with Tier Advancements")
    ax3.legend(fontsize=9)
    ax3.set_ylim(0, 1)
    ax3.grid(True, alpha=0.3)

    # Right: tier distribution pie chart
    ax4 = axes2[1]
    from collections import Counter
    tier_counts = Counter(tiers)
    tier_order  = [t for t in ["novice","apprentice","practitioner","expert","master"]
                   if t in tier_counts]
    counts      = [tier_counts[t] for t in tier_order]
    colors      = [tier_colors.get(t, "#ccc") for t in tier_order]
    ax4.pie(counts, labels=tier_order, colors=colors, autopct="%1.0f%%",
            startangle=90, textprops={"fontsize": 9})
    ax4.set_title("Episodes Spent Per Difficulty Tier")

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/curriculum_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"   Chart saved: {SAVE_DIR}/curriculum_curve.png")
    print(f"   Tier advancements: {len(advancements)}")
    for ep, tier in advancements:
        print(f"     Episode {ep}: advanced to {tier}")

except Exception as e:
    print(f"   Could not fetch curriculum data: {e}")
    print("   (This is fine — the reward curve above is the primary evidence)")

## Cell 10 — Before vs After Demo

Runs the same procurement scenario with both the untrained base model and the trained model,
side by side. This is the qualitative demonstration of what training achieved.

The untrained model has no concept of procurement strategy — it tends to repeat the same
action type (e.g. `raise_pr`) without price or quantity, which the environment penalises.
The trained model should show more diverse, strategically coherent actions and higher rewards.

**Note:** The before/after comparison is most compelling after a full training run
(2116 samples, 3 epochs). After a short dry run, both agents may look similar.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, json

def run_inference(model, tokenizer, obs, max_new_tokens=300):
    """
    Run one inference step. Returns (action_dict, raw_text).
    Invalid JSON returns (None, raw_text) — the caller falls back to a defer action.
    """
    prompt = obs_to_prompt(obs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    # Strip markdown fences if the model wrapped its output
    clean = generated
    if clean.startswith("```"):
        parts = clean.split("```")
        clean = parts[1] if len(parts) > 1 else clean
        if clean.startswith("json"):
            clean = clean[4:].strip()

    try:
        action = json.loads(clean)
        # Validate and correct IDs against the current observation
        obs_sups  = {s["supplier_id"] for s in obs["suppliers"]}
        obs_items = {r["item_id"] for r in obs["requirements"]}
        if "supplier_id" not in action or action["supplier_id"] not in obs_sups:
            action["supplier_id"] = obs["suppliers"][0]["supplier_id"]
        if "item_id" not in action or action["item_id"] not in obs_items:
            action["item_id"] = obs["requirements"][0]["item_id"]
        if "message" not in action:
            action["message"] = "Proceeding with action"
        return action, generated
    except Exception as e:
        print(f"    [parse error: {e}] raw: {generated[:100]}")
        return None, generated


def run_demo_episode(model, tokenizer, task_id="easy_negotiation",
                     seed=999, max_steps=8, label="Agent"):
    """Run a short demo episode and print step-by-step decisions."""
    reset_data = env_reset(task_id, seed=seed)
    obs        = reset_data["observation"]
    actions_taken = []
    total_reward  = 0

    print(f"\n{'─'*60}")
    print(f"{label} — {task_id} (seed={seed})")
    print(f"Week 1/{obs['total_weeks']} | Budget: ${obs['budget_remaining']:,.0f}")
    print(f"{'─'*60}")

    for step_i in range(max_steps):
        if obs.get("done", False):
            break

        action, raw = run_inference(model, tokenizer, obs)

        if action is None:
            print(f"  Step {step_i+1}: ❌ Invalid JSON — falling back to defer")
            action = {
                "action_type": "defer",
                "supplier_id": obs["suppliers"][0]["supplier_id"],
                "item_id":     obs["requirements"][0]["item_id"],
                "message":     "Deferring",
            }

        step_data    = env_step(action)
        reward       = step_data["reward"]["total"]
        total_reward += reward
        obs          = step_data["observation"]

        print(f"  Step {step_i+1}: {action['action_type']:20s} | "
              f"reward={reward:.4f} | "
              f"{step_data['reward'].get('explanation', '')[:45]}")
        if action.get("notes"):
            print(f"          Notes: {action['notes'][:60]}")

        actions_taken.append({
            "step": step_i + 1, "action": action["action_type"],
            "reward": reward,   "notes":  action.get("notes", ""),
        })

    avg_reward = total_reward / max(len(actions_taken), 1)
    print(f"\n  Avg reward: {avg_reward:.4f}")
    return actions_taken, avg_reward


DEMO_SEED  = 777
DEMO_STEPS = 6

print("=" * 60)
print("BEFORE vs AFTER TRAINING COMPARISON")
print("=" * 60)

# Load the untrained base model for comparison
print("\nLoading untrained base model...")
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", load_in_4bit=True
)
base_actions, base_score = run_demo_episode(
    base_model, base_tokenizer,
    seed=DEMO_SEED, max_steps=DEMO_STEPS, label="❌ UNTRAINED AGENT"
)
del base_model
torch.cuda.empty_cache()

# Run the trained model on the same scenario
print()
trained_actions, trained_score = run_demo_episode(
    model, tokenizer,
    seed=DEMO_SEED, max_steps=DEMO_STEPS, label="✅ TRAINED AGENT"
)

print()
print("=" * 60)
print("RESULTS")
print("=" * 60)
improvement = ((trained_score - base_score) / max(base_score, 1e-6)) * 100
print(f"  Untrained avg reward: {base_score:.4f}")
print(f"  Trained   avg reward: {trained_score:.4f}")
print(f"  Improvement:          +{improvement:.1f}%")
print("=" * 60)

## Cell 11 — Train on Medium Task (Optional)

Continue training on `medium_adversarial` after the easy task is complete.
The medium task introduces:
- 12 suppliers with 3 deceptive agents hidden in the pool
- A rule-based rival buyer competing for the same hardware capacity
- A supplier that goes dark (stops responding) at week 5

Training continues from the easy-trained model weights (curriculum learning).
A lower learning rate is used to preserve what was learned on the easy task.

**Estimated time:** ~45 min for episode generation + ~3 hrs training on T4.

In [ ]:
# Generate medium episodes
print("Generating medium_adversarial episodes...")
medium_dataset, medium_scores = generate_episodes(
    task_id="medium_adversarial",
    n_episodes=MEDIUM_EPISODES,
    max_steps=MAX_STEPS_PER_EP,
    seed=100,
)

medium_hf_dataset = Dataset.from_list([
    {"prompt": ep["prompt"]} for ep in medium_dataset
])

medium_config = GRPOConfig(
    learning_rate=LEARNING_RATE / 2,   # lower LR for continued training
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=MAX_COMPLETION_LEN,
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION_LEN,
    output_dir=f"{SAVE_DIR}/medium",
    logging_steps=LOGGING_STEPS,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    fp16=True,
    bf16=False,
)

medium_trainer = GRPOTrainer(
    model=model,
    reward_funcs=procurement_reward_fn,
    args=medium_config,
    train_dataset=medium_hf_dataset,
    processing_class=tokenizer,
)

print("\n🚀 Training on medium_adversarial...")
medium_trainer.train()
model.save_pretrained(f"{SAVE_DIR}/medium_final")
print("✅ Medium training complete")

## Cell 12 — Push Model to HuggingFace Hub (Optional)

Publishes the trained LoRA adapter to your HuggingFace profile.
Run this after the full training run to make the model publicly accessible.

The model will be available at:
`https://huggingface.co/prasanthdj8/negotiateai-procurement-agent`

In [ ]:
# ── Step 1: Save LoRA adapter locally (always run this) ───────────────────────
# Saves the adapter weights only — lightweight, fast, correct for QLoRA.
# Do NOT merge a 4-bit model naively — it damages model quality (FAQ Q8/Q16).
print("Saving LoRA adapter...")
model.save_pretrained(f"{SAVE_DIR}/adapter")
tokenizer.save_pretrained(f"{SAVE_DIR}/adapter")
print(f"✅ Adapter saved: {SAVE_DIR}/adapter")

# ── Step 2: Save merged 16-bit model (for inference + demo) ───────────────────
# This is the correct merge path — dequantise first, then merge LoRA.
# Required for the before/after demo in Cell 10 to work correctly.
print("\nMerging and saving 16-bit model for inference...")
try:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    import torch, gc

    # Load fresh base model in float16 (NOT 4-bit) for a clean merge
    base_fp16 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    # Load adapter and merge — this is the safe path
    merged = PeftModel.from_pretrained(base_fp16, f"{SAVE_DIR}/adapter")
    merged = merged.merge_and_unload()   # proper merge, no quality loss

    merged.save_pretrained(f"{SAVE_DIR}/merged_16bit")
    tokenizer.save_pretrained(f"{SAVE_DIR}/merged_16bit")
    print(f"✅ Merged model saved: {SAVE_DIR}/merged_16bit")

    # Immediately test inference to confirm the merge worked
    test_in = tokenizer("Test procurement action", return_tensors="pt").to(merged.device)
    with torch.no_grad():
        out = merged.generate(**test_in, max_new_tokens=20,
                               pad_token_id=tokenizer.eos_token_id)
    print(f"   Inference test: ✅ ({out.shape[1]} tokens generated)")

    del base_fp16, merged
    gc.collect()
    if hasattr(torch.cuda, "empty_cache"):
        torch.cuda.empty_cache()

except Exception as e:
    print(f"   Merge failed: {e}")
    print("   ⚠️  Use adapter directly for demo:")
    print(f"       model = PeftModel.from_pretrained(base, '{SAVE_DIR}/adapter')")

# ── Step 3: Push to HuggingFace Hub (optional) ────────────────────────────────
if HF_TOKEN and HF_TOKEN != "your-hf-token-here":
    from huggingface_hub import login
    login(token=HF_TOKEN)

    model.push_to_hub("prasanthdj8/negotiateai-procurement-agent", token=HF_TOKEN)
    tokenizer.push_to_hub("prasanthdj8/negotiateai-procurement-agent", token=HF_TOKEN)
    print("\n✅ Adapter pushed to HuggingFace Hub")
    print("   https://huggingface.co/prasanthdj8/negotiateai-procurement-agent")
else:
    print("\n⚠️  HF_TOKEN not set — skipping Hub push")
    print("   Set HF_TOKEN in Cell 2 and re-run if needed")


## Cell 13 — Pitch Day Checklist

Run this immediately before presenting. All five checks must be green before going on stage.

In [ ]:
import os
import numpy as np

print("=" * 60)
print("PITCH DAY CHECKLIST")
print("=" * 60)

checks = []

healthy     = env_health()
model_saved = os.path.exists(f"{SAVE_DIR}/final/adapter_config.json")
curve_saved = os.path.exists(f"{SAVE_DIR}/reward_curve.png")
trained     = len(grpo_step_rewards) > 0

checks.append(("HF Space live & healthy",    healthy))
checks.append(("Trained model saved locally", model_saved))
checks.append(("Reward curve chart saved",    curve_saved))
checks.append(("GRPO training completed",     trained))

if grpo_step_rewards and len(grpo_step_rewards) >= 40:
    first = np.mean(grpo_step_rewards[:20])
    last  = np.mean(grpo_step_rewards[-20:])
    checks.append((f"Reward improved ({first:.3f} → {last:.3f})", last > first))

print()
for label, passed in checks:
    print(f"  {'✅' if passed else '❌'}  {label}")

print()
all_passed = all(p for _, p in checks)
if all_passed:
    print("🎉 All checks passed — you're ready to pitch!")
else:
    failed = [l for l, p in checks if not p]
    print(f"⚠️  {len(failed)} check(s) need attention before pitching.")

if grpo_step_rewards and len(grpo_step_rewards) >= 40:
    first = np.mean(grpo_step_rewards[:20])
    last  = np.mean(grpo_step_rewards[-20:])
    pct   = ((last - first) / max(first, 1e-6)) * 100
    print(f"\nPitch line:")
    print(f'  "Our trained agent improved from {first:.2f} to {last:.2f} — '
          f'a {pct:.0f}% improvement — trained entirely from environment reward signals, no human labels."')